In [2]:
import pickle
import numpy as np
from tqdm import tqdm
from rdkit import Chem
from sklearn.model_selection import train_test_split

np.random.seed(0)


with open('char2idx_class_V1.pkl','rb') as f:
    class_  = pickle.load(f)
with open('char2idx_super_V1.pkl','rb') as f:
    superclass_  = pickle.load(f)
with open('char2idx_path_V1.pkl','rb') as f:
    pathway_  = pickle.load(f)

with open('datset_class_all_V1.pkl','rb') as r:
    dataset = pickle.load(r)
    
# Train, Validation, and test set 

b_key = list(dataset.keys())
np.random.shuffle(b_key)
dict_ = np.array(b_key)
Y_ = np.array([ np.max(np.where(dataset[i]['Class']==1)[0]) for i in dict_])

train_D, test_dict, y_train, y_test = train_test_split(dict_, Y_, test_size=0.2, random_state=1, stratify = Y_)
train_dict, val_dict, y_train, y_val = train_test_split(train_D, y_train, test_size=0.2, random_state=1, stratify = y_train)

#Implement data augmentation
aug = {}
for i in tqdm(dict_):
    smiles = dataset[i]['SMILES']
    ori_path = dataset[i]['Pathway']
    ori_sup = dataset[i]['Super_class']
    ori_class = dataset[i]['Class']
    inchi_key = Chem.inchi.MolToInchiKey(Chem.MolFromSmiles(smiles))[:14]
    aug[i] = {'SMILES':smiles,'Pathway':ori_path,'Super_class':ori_sup,'Class':ori_class}        


100%|██████████| 77705/77705 [00:59<00:00, 1308.56it/s]


In [3]:
columns = ["key", "SMILES"] + list(pathway_.keys()) + list(superclass_.keys()) + list(class_.keys())

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Train, Validation, and test set

b_key = list(dataset.keys())
np.random.shuffle(b_key)
dict_ = np.array(b_key)
Y_ = np.array([ np.max(np.where(dataset[i]['Class']==1)[0]) for i in dict_])

In [6]:
train_D

array(['PTNBEYQHWBDDAI', 'PHBDYBJOGVFIQU', 'DXUCGAHPDLXISA', ...,
       'JCOWERMIDAWOPQ', 'PIGOPELHGLPKLL', 'ZQVQWQOKMMAWIW'], dtype='<U14')

In [13]:
val_dict.shape

(12433,)

In [4]:
dataset["CKDDRHZIAZRDBW"]["Class"].shape

(653,)

In [7]:
import pandas as pd

new_dataframe = pd.DataFrame(columns=columns)

dataframe_dict = {}
new_row = {"key": [], 'SMILES':[]}
for j in pathway_:
    new_row[j] = []
for j in superclass_:
    new_row[j] = []
for j in class_:
    new_row[j] = []

for i in tqdm(aug):
    new_row["key"].append(i)
    new_row["SMILES"].append(aug[i]['SMILES'])
    for j in pathway_:
        new_row[j].append(aug[i]['Pathway'][pathway_[j]])
    for j in superclass_:
        new_row[j].append(aug[i]['Super_class'][superclass_[j]])
    for j in class_:
        new_row[j].append(aug[i]['Class'][class_[j]])


100%|██████████| 77705/77705 [00:21<00:00, 3565.13it/s]


In [8]:
new_dataframe = pd.DataFrame().from_dict(new_row)
new_dataframe

In [ ]:
train_dataset, test_dataset = train_test_split(new_dataframe, test_size=0.2, random_state=1, stratify = Y_)
train_dataset, val_dataset = train_test_split(train_dataset, test_size=0.2, random_state=1, stratify = y_train)

In [ ]:

train_dataset.to_csv("train_dataset.csv", index=False)
test_dataset.to_csv("test_dataset.csv", index=False)
val_dataset.to_csv("validation_dataset.csv", index=False)